
# **EXTRACCIÓN DE COMENTARIOS DE YOUTUBE A CSV**
# **Proyecto: Detección de posible desinformación política**
# **Fuente: Youtube**
# **API usada: YouTube Data API v3**


## 1. INSTALACIÓN DE LIBRERÍAS

In [1]:
!pip install google-api-python-client pandas

## 2. IMPORTACIÓN DE LIBRERÍAS

In [2]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
import re
import hashlib
import time
import os
from datetime import datetime

## 3. CONFIGURACIÓN GENERAL

In [ ]:
#Llave de la API
API_KEY = ""

# Cantidad deseada por video
MAX_COMENTARIOS_POR_VIDEO = 300

# Cada llamada permite máximo 100 comentarios
MAX_RESULTADOS_POR_PAGINA = 100

# Pausa ética entre llamadas
PAUSA_ENTRE_PAGINAS = 0.5
PAUSA_ENTRE_VIDEOS = 1

# Carpeta de salida
CARPETA_SALIDA = "data/raw"

# Crear carpeta si no existe
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Validación de API Key
if API_KEY == "" or API_KEY == "PEGA_AQUI_TU_API_KEY":
    raise ValueError("Debes pegar tu API Key en la variable API_KEY antes de ejecutar.")

# Crear cliente de YouTube
youtube = build("youtube", "v3", developerKey=API_KEY)

## 4. LINKS DE VIDEOS

In [4]:
video_urls = [
    "https://www.youtube.com/live/-ePoQCG_IVs?si=9DQZErg9rlkX5DEL",
    "https://www.youtube.com/live/Vo70i88M-Hg?si=dKh1ztNEC9j3ExoT",
    "https://www.youtube.com/live/UgH5IjXGR8w?si=YUz2RxLXamCkp6xH",
    "https://www.youtube.com/live/kzwAQuQxd3A?si=ThEPAJqlhDfbyqFW",
    "https://www.youtube.com/live/bY61pgGHcPw?si=tNAM2FAPdCTFMMA2"
]

## 5. FUNCIÓN PARA EXTRAER ID DEL VIDEO

In [5]:
def extraer_video_id(url):
    """
    Extrae el ID de un video de YouTube desde distintos formatos de URL.
    """

    patrones = [
        r"v=([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"youtube\.com/live/([^?&]+)",
        r"youtube\.com/embed/([^?&]+)"
    ]

    for patron in patrones:
        match = re.search(patron, url)
        if match:
            return match.group(1)

    return None

## 6. FUNCIÓN PARA ANONIMIZAR AUTORES

In [6]:
def anonimizar_autor(nombre):
    """
    Convierte el nombre visible del autor en un código anónimo.
    """

    if nombre is None:
        nombre = "desconocido"

    codigo = hashlib.sha256(nombre.encode("utf-8")).hexdigest()[:10]
    return f"usuario_anon_{codigo}"

## 7. FUNCIÓN PARA OBTENER INFORMACIÓN DEL VIDEO

In [7]:
def obtener_info_video(video_id):
    """
    Obtiene metadatos básicos del video y registra el estado de la consulta.
    """

    try:
        request = youtube.videos().list(
            part="snippet,statistics",
            id=video_id
        )

        response = request.execute()

        # La consulta a la API funcionó, pero no devolvió videos.
        # Técnicamente la respuesta fue correcta, pero el recurso no se encontró.
        if not response.get("items"):
            return {
                "video_id": video_id,
                "titulo_video": None,
                "canal": None,
                "fecha_video": None,
                "vistas_video": None,
                "likes_video": None,
                "estado_video": "no encontrado",
                "codigo_http": 200,
                "mensaje_error": "La API respondió correctamente, pero no encontró el video."
            }

        item = response["items"][0]
        snippet = item.get("snippet", {})
        stats = item.get("statistics", {})

        return {
            "video_id": video_id,
            "titulo_video": snippet.get("title"),
            "canal": snippet.get("channelTitle"),
            "fecha_video": snippet.get("publishedAt"),
            "vistas_video": stats.get("viewCount"),
            "likes_video": stats.get("likeCount"),
            "estado_video": "encontrado",
            "codigo_http": 200,
            "mensaje_error": None
        }

    except HttpError as e:
        codigo_error = e.resp.status if e.resp else None

        print(f"Error al obtener información del video {video_id}:")
        print(f"Código HTTP: {codigo_error}")
        print(e)

        return {
            "video_id": video_id,
            "titulo_video": None,
            "canal": None,
            "fecha_video": None,
            "vistas_video": None,
            "likes_video": None,
            "estado_video": "error",
            "codigo_http": codigo_error,
            "mensaje_error": str(e)
        }

## 8. FUNCIÓN PARA EXTRAER COMENTARIOS

In [8]:
def extraer_comentarios_video(video_id, url_video, max_comentarios=300):
    """
    Extrae hasta max_comentarios comentarios principales de un video.
    Si el video tiene menos comentarios disponibles, extrae los que existan.
    """

    comentarios = []
    next_page_token = None
    total_extraidos = 0

    info_video = obtener_info_video(video_id)

    while total_extraidos < max_comentarios:

        try:
            comentarios_restantes = max_comentarios - total_extraidos

            max_resultados = min(
                MAX_RESULTADOS_POR_PAGINA,
                comentarios_restantes
            )

            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=max_resultados,
                pageToken=next_page_token,
                textFormat="plainText",
                order="relevance"
            )

            response = request.execute()

            items = response.get("items", [])

            if not items:
                break

            for item in items:
                top_comment = item["snippet"]["topLevelComment"]["snippet"]

                autor_original = top_comment.get("authorDisplayName")
                autor_anonimo = anonimizar_autor(autor_original)

                comentarios.append({
                    "video_id": video_id,
                    "url_video": url_video,
                    "titulo_video": info_video["titulo_video"],
                    "canal": info_video["canal"],
                    "fecha_video": info_video["fecha_video"],
                    "vistas_video": info_video["vistas_video"],
                    "likes_video": info_video["likes_video"],
                    "comentario_id": item["snippet"]["topLevelComment"]["id"],
                    "autor_anonimo": autor_anonimo,
                    "texto_comentario": top_comment.get("textDisplay"),
                    "likes_comentario": top_comment.get("likeCount"),
                    "fecha_comentario": top_comment.get("publishedAt"),
                    "total_respuestas": item["snippet"].get("totalReplyCount"),
                    "fuente": "YouTube",
                    "metodo_extraccion": "YouTube Data API v3",
                    "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })

            total_extraidos = len(comentarios)

            next_page_token = response.get("nextPageToken")

            if not next_page_token:
                break

            time.sleep(PAUSA_ENTRE_PAGINAS)

        except HttpError as e:
            print(f"No se pudieron extraer comentarios del video {video_id}.")
            print(e)
            break

    return comentarios, info_video

## 9. EXTRACCIÓN GENERAL

In [9]:
video_ids = [extraer_video_id(url) for url in video_urls]

print("IDs de videos extraídos:")
print(video_ids)

todos_los_comentarios = []
log_extraccion = []

for url_video, video_id in zip(video_urls, video_ids):

    if video_id is None:
        log_extraccion.append({
            "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "fuente": "YouTube",
            "metodo": "YouTube Data API v3",
            "video_id": None,
            "url_video": url_video,
            "titulo_video": None,
            "comentarios_extraidos": 0,
            "comentarios_objetivo": MAX_COMENTARIOS_POR_VIDEO,
            "estado": "error",
            "observacion": "No se pudo extraer el ID del video"
        })
        continue

    print(f"\nExtrayendo comentarios del video: {video_id}")

    comentarios, info_video = extraer_comentarios_video(
        video_id=video_id,
        url_video=url_video,
        max_comentarios=MAX_COMENTARIOS_POR_VIDEO
    )

    todos_los_comentarios.extend(comentarios)

    cantidad = len(comentarios)

    print(f"Comentarios extraídos de este video: {cantidad}")

    if cantidad == 0:
        estado = "sin comentarios o acceso limitado"
    elif cantidad < MAX_COMENTARIOS_POR_VIDEO:
        estado = "extraccion parcial"
    else:
        estado = "exitoso"

    log_extraccion.append({
    "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "fuente": "YouTube",
    "metodo": "YouTube Data API v3",
    "video_id": video_id,
    "url_video": url_video,
    "titulo_video": info_video["titulo_video"],
    "canal": info_video["canal"],
    "codigo_http": info_video.get("codigo_http"),
    "estado_video": info_video.get("estado_video"),
    "mensaje_error": info_video.get("mensaje_error"),
    "comentarios_extraidos": cantidad,
    "comentarios_objetivo": MAX_COMENTARIOS_POR_VIDEO,
    "estado": estado,
    "observacion": "Extracción de comentarios públicos mediante API oficial"
    })

    time.sleep(PAUSA_ENTRE_VIDEOS)

IDs de videos extraídos:
['-ePoQCG_IVs', 'Vo70i88M-Hg', 'UgH5IjXGR8w', 'kzwAQuQxd3A', 'bY61pgGHcPw']

Extrayendo comentarios del video: -ePoQCG_IVs
Comentarios extraídos de este video: 300

Extrayendo comentarios del video: Vo70i88M-Hg
Comentarios extraídos de este video: 300

Extrayendo comentarios del video: UgH5IjXGR8w
Comentarios extraídos de este video: 175

Extrayendo comentarios del video: kzwAQuQxd3A
Comentarios extraídos de este video: 300

Extrayendo comentarios del video: bY61pgGHcPw
Comentarios extraídos de este video: 263


## 10. CREACIÓN DE DATAFRAMES

In [10]:
df_comentarios = pd.DataFrame(todos_los_comentarios)
df_log = pd.DataFrame(log_extraccion)

total_videos = len(video_ids)
total_comentarios = len(df_comentarios)

df_resumen = pd.DataFrame([{
    "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "fuente": "YouTube",
    "metodo": "YouTube Data API v3",
    "videos_procesados": total_videos,
    "comentarios_objetivo_por_video": MAX_COMENTARIOS_POR_VIDEO,
    "comentarios_totales_extraidos": total_comentarios,
    "archivo_comentarios": "comentarios_youtube_raw.csv",
    "archivo_log": "log_extraccion_youtube.csv"
}])

## 11. GUARDAR ARCHIVOS CSV

In [11]:
ruta_comentarios = f"{CARPETA_SALIDA}/comentarios_youtube_raw.csv"
ruta_log = f"{CARPETA_SALIDA}/log_extraccion_youtube.csv"
ruta_resumen = f"{CARPETA_SALIDA}/resumen_extraccion_youtube.csv"

df_comentarios.to_csv(
    ruta_comentarios,
    index=False,
    encoding="utf-8-sig"
)

df_log.to_csv(
    ruta_log,
    index=False,
    encoding="utf-8-sig"
)

df_resumen.to_csv(
    ruta_resumen,
    index=False,
    encoding="utf-8-sig"
)

## 12. VERIFICACIÓN FINAL

In [12]:
print("\n====================================")
print("EXTRACCIÓN FINALIZADA")
print("====================================")
print("Videos procesados:", total_videos)
print("Comentarios totales extraídos:", total_comentarios)
print("CSV principal:", ruta_comentarios)
print("Log de extracción:", ruta_log)
print("Resumen de extracción:", ruta_resumen)
print("====================================")

print("\nResumen por video:")
display(df_log)

print("\nPrimeros comentarios extraídos:")
display(df_comentarios.head(10))


EXTRACCIÓN FINALIZADA
Videos procesados: 5
Comentarios totales extraídos: 1338
CSV principal: data/raw/comentarios_youtube_raw.csv
Log de extracción: data/raw/log_extraccion_youtube.csv
Resumen de extracción: data/raw/resumen_extraccion_youtube.csv

Resumen por video:


,fecha_extraccion,fuente,metodo,video_id,url_video,titulo_video,canal,codigo_http,estado_video,mensaje_error,comentarios_extraidos,comentarios_objetivo,estado,observacion
0,2026-06-17 16:38:01,YouTube,YouTube Data API v3,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,200,encontrado,None,300,300,exitoso,Extracción de comentarios públicos mediante AP...
1,2026-06-17 16:38:04,YouTube,YouTube Data API v3,Vo70i88M-Hg,https://www.youtube.com/live/Vo70i88M-Hg?si=dK...,¡Gran fraude! #fraudismo #keiko #hildebrandt #...,Hildebrandt en sus trece,200,encontrado,None,300,300,exitoso,Extracción de comentarios públicos mediante AP...
2,2026-06-17 16:38:06,YouTube,YouTube Data API v3,UgH5IjXGR8w,https://www.youtube.com/live/UgH5IjXGR8w?si=YU...,RAFAEL LÓPEZ ALIAGA VUELVE A LA MUNICIPALIDAD ...,El diario de Curwen,200,encontrado,None,175,300,extraccion parcial,Extracción de comentarios públicos mediante AP...
3,2026-06-17 16:38:09,YouTube,YouTube Data API v3,kzwAQuQxd3A,https://www.youtube.com/live/kzwAQuQxd3A?si=Th...,JORGE NIETO EN BRUTALIDAD POLÍTICA | #BRUTALID...,El diario de Curwen,200,encontrado,None,300,300,exitoso,Extracción de comentarios públicos mediante AP...
4,2026-06-17 16:38:12,YouTube,YouTube Data API v3,bY61pgGHcPw,https://www.youtube.com/live/bY61pgGHcPw?si=tN...,JEE RECHAZA PEDIDO DE JP PARA ANULAR MESAS EN ...,La República - LR+,200,encontrado,None,263,300,extraccion parcial,Extracción de comentarios públicos mediante AP...



Primeros comentarios extraídos:


,video_id,url_video,titulo_video,canal,fecha_video,vistas_video,likes_video,comentario_id,autor_anonimo,texto_comentario,likes_comentario,fecha_comentario,total_respuestas,fuente,metodo_extraccion,fecha_extraccion
0,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgzZxWFbr-Hx0YG7uap4AaABAg,usuario_anon_8871bf5dc9,"Ese es el Problema de la Dictadura,criminal",109,2026-06-16T10:29:22Z,12,YouTube,YouTube Data API v3,2026-06-17 16:37:58
1,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgwVYv0jPfFJNOKHLcV4AaABAg,usuario_anon_563d8ce904,Siga asi don Cesar...este canal tambien debe s...,18,2026-06-16T16:21:15Z,0,YouTube,YouTube Data API v3,2026-06-17 16:37:58
2,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgxFkjI7d18SHtj581h4AaABAg,usuario_anon_0c6481592f,"Un pueblo que elige a corruptos,, inmorales, l...",536,2026-06-16T06:37:31Z,66,YouTube,YouTube Data API v3,2026-06-17 16:37:58
3,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgwDNl6lX1wptZsjCAR4AaABAg,usuario_anon_d52f12c3e4,Gracias don César Hildebrant por tan excelentí...,168,2026-06-16T12:02:23Z,4,YouTube,YouTube Data API v3,2026-06-17 16:37:58
4,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,Ugz-nKAqRh6lLNxy5PR4AaABAg,usuario_anon_93ccc36b89,Ni ell gobernador de Huancavelica se pronunció...,107,2026-06-16T11:42:38Z,2,YouTube,YouTube Data API v3,2026-06-17 16:37:58
5,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgxgxBoxzeLp6KFC4yV4AaABAg,usuario_anon_760eee18f7,Bravo César!!!! No nos vamos a callar!!!!!!!!👍👍👍,260,2026-06-16T10:35:34Z,9,YouTube,YouTube Data API v3,2026-06-17 16:37:58
6,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgxRSt9Vcr2RlTHOCbt4AaABAg,usuario_anon_fdaa97653a,Fui personero del casquito. Jamás me equivoqué,142,2026-06-16T06:56:47Z,3,YouTube,YouTube Data API v3,2026-06-17 16:37:58
7,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgyavITT8sN6B_EV2fx4AaABAg,usuario_anon_39d448b2e3,QUE DIOS NOS AMPARE!,152,2026-06-16T10:26:15Z,13,YouTube,YouTube Data API v3,2026-06-17 16:37:58
8,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgwLBcAcwIKoW1_ZyYV4AaABAg,usuario_anon_c0eb6b3e21,EL MEJOR TRUCO DEL DEMONIO ES HACERNOS CREER Q...,5,2026-06-16T22:32:02Z,0,YouTube,YouTube Data API v3,2026-06-17 16:37:58
9,-ePoQCG_IVs,https://www.youtube.com/live/-ePoQCG_IVs?si=9D...,Se viene la dictadura #hildebrandt #keiko,Hildebrandt en sus trece,2026-06-16T03:29:38Z,204300,14452,UgwfVD6F35kAk3S8mPt4AaABAg,usuario_anon_7b79ece977,A seguir luchando señor Cesar,13,2026-06-16T13:45:39Z,0,YouTube,YouTube Data API v3,2026-06-17 16:37:58
